# Loading DGCI embeddings:

In [ ]:
import numpy as np                                          # already imported in cell 1
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as shc
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import pairwise_distances
import umap

codes_path = "/home/ids/gmargari-24/airway_project/DGCI/logs/unregistered_128_2M_4k/latent_codes.npy"
enc_output = np.load(codes_path).astype(np.float64)         # (419, 128); float64 keeps ward's sums stable

# dendrogram, Fig. 4

In [ ]:
plt.rcParams['lines.linewidth'] = 20
fig = plt.figure(figsize=(100, 60))
fig.patch.set_facecolor('slateblue')
plt.xlabel("Cluster Index", fontsize=250, color='white')
plt.ylabel("Euclidean Distances", fontsize=250, color='white')

# --- compute linkage once, and derive the 2-cluster line height from the data ---
Z = shc.linkage(enc_output, method='ward')
thr = (Z[-1, 2] + Z[-2, 2]) / 2          # midpoint between the last two merges = 2-cluster cut

dend = shc.dendrogram(Z,
                      truncate_mode  ='lastp',  # show only the last p merged clusters
                      p              =30,  # show only the last p merged clusters
                      leaf_rotation  =90.,
                      leaf_font_size =12.,
                      show_contracted=True,
                      # color_threshold=5500,
                      )
ax = plt.gca()
ax.tick_params(axis='x', which='major', labelsize=150, colors='white')
ax.tick_params(axis='y', which='major', labelsize=150, colors='white')

# Add a horizontal line for 2 clusters
ax.axhline(y=thr, linestyle='--', color='red', linewidth=15)

# Get the x-positions of the vertical lines
x_vals = dend['icoord']

# Iterate over the x-positions and find the intersection points with the horizontal line
for i, x in enumerate(x_vals):
    for j in range(len(x)):
        y = dend['dcoord'][i][j]
        if abs(y - thr) < 1e-5:  # check if the y-coordinate is close to the horizontal line
            ax.annotate('', xy=(x[j], y), xytext=(x[j], y + 20), arrowprops=dict(facecolor='white', edgecolor='white', arrowstyle='-|>,head_width=1,head_length=3'), fontsize=150)

plt.savefig('/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/HC.png')
# plt.show()

# average intra-cluster distance, Eq. 1

In [ ]:
avg_dist = []
for c in range(1, 50):
  cluster = AgglomerativeClustering(n_clusters=c, metric='euclidean', linkage='ward')
  members = cluster.fit_predict(enc_output)
  k = np.unique(members)
  num_clusters = k.shape[0]
  sum = 0
  for cluster in range(num_clusters):
    cluster_sum = 0  
    cluster_sum = pairwise_distances(enc_output[members == cluster]).mean()  
    sum += cluster_sum
  sum = sum/num_clusters
  avg_dist.append(sum)

print(avg_dist)

In [ ]:
distance_txt = np.savetxt("/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/avg_dist0.txt", avg_dist)

In [ ]:
avg_dist = np.loadtxt('/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/avg_dist0.txt')
# avg_dist = avg_dist.astype(int) # for autoencoder only

In [ ]:
plt.rcParams['lines.linewidth'] = 10

fig = plt.figure(figsize=(30, 18))

fig.patch.set_facecolor('slateblue')

plt.xlabel("Number of Phenotypes", fontsize=50, color='white')
plt.ylabel("Intra-cluster Distance", fontsize=50, color='white')

plt.plot(range(1, 50), avg_dist, color='mediumvioletred')

# Show every 2nd value
plt.xticks(range(1, 50, 2))

ax = plt.gca()
ax.set_facecolor('midnightblue')

ax.tick_params(axis='x', labelsize=25, colors='white')
ax.tick_params(axis='y', labelsize=25, colors='white')

plt.tight_layout()

plt.savefig(
    '/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/elbow0.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
reduction = np.diff(avg_dist)
plt.figure(figsize=(30, 18))
plt.plot(range(2, 50),
        reduction,
        linewidth=10,
        color='mediumvioletred')

plt.xlabel("Number of Phenotypes", fontsize=50, color='white')
plt.ylabel("Reduction in Intra-cluster Distance", fontsize=50, color='white')
plt.xticks(range(2, 50, 2))
ax = plt.gca()
ax.set_facecolor('midnightblue')
ax.figure.patch.set_facecolor('slateblue')
ax.tick_params(axis='x', labelsize=25, colors='white')
ax.tick_params(axis='y', labelsize=25, colors='white')
plt.tight_layout()
plt.savefig('/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/reduction.png',dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
test1 = []
for i in range(1,19):    
    test = (avg_dist[i-1] - avg_dist[i]) / avg_dist[i-1]
    test1.append(test)

print(test1)

# top 3 reductions, largest first
order = sorted(range(len(test1)), key=lambda j: test1[j], reverse=True)
top3 = order[:3]
for rank, j in enumerate(top3, 1):
    k = j + 2                      # test1[j] is the drop into k = j+2 clusters
    print(f"#{rank}: k={k}  reduction={test1[j]:.4f}")

# final clustering — they hardcoded n_clusters=2; change this to the k printed by Cell 28 if it differs)

In [ ]:
cluster = AgglomerativeClustering(n_clusters=4, metric='euclidean', linkage='ward')
members = cluster.fit_predict(enc_output)

# label assignment

In [ ]:
labels = members

# save labels

In [ ]:
labelstxt = np.savetxt("/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/labels.txt", labels)

# UMAP

In [ ]:
# enc_output = (N subjects, latent dimensions)
# labels = cluster assignment for each subject

print("Embedding shape:", enc_output.shape)
print("Labels shape:", labels.shape)
print("Clusters:", np.unique(labels))

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

min_dist = 0.0125
output_dir = (
    "/home/ids/gmargari-24/airway_project/DGCI/logs/"
    "binary_128_2M_4k"
)

# n_neighbors: 5, 15, 25, ..., 395
neighbor_values = range(5, 55, 5)

# ------------------------------------------------------------
# Generate UMAP projections
# ------------------------------------------------------------

unique_labels = np.unique(labels)

for n_neighbors in neighbor_values:

    print(f"Running UMAP with n_neighbors = {n_neighbors}...")

    # --------------------------------------------------------
    # UMAP
    # --------------------------------------------------------
    umap_reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=2,
        metric="euclidean",
        random_state=42
    )

    umap_embedding = umap_reducer.fit_transform(enc_output)

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------
    plt.figure(figsize=(10, 8))

    scatter = plt.scatter(
        umap_embedding[:, 0],
        umap_embedding[:, 1],
        c=labels,
        cmap="tab10",
        s=35,
        alpha=0.85
    )

    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.title(
        f"UMAP projection of airway latent representations\n"
        f"n_neighbors = {n_neighbors}"
    )

    # --------------------------------------------------------
    # Legend
    # --------------------------------------------------------
    handles, _ = scatter.legend_elements()

    plt.legend(
        handles,
        [f"Cluster {c}" for c in unique_labels],
        title="Phenotype"
    )

    plt.tight_layout()

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------
    save_path = (
        f"{output_dir}/UMAP_clusters_neighbors_{n_neighbors}.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Saved: {save_path}")

print("All UMAP projections generated.")

In [ ]:
from sklearn.manifold import TSNE

# enc_output = (N subjects, latent dimensions)
# labels = cluster assignment for each subject

print("Embedding shape:", enc_output.shape)
print("Labels shape:", labels.shape)
print("Clusters:", np.unique(labels))

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

output_dir = (
    "/home/ids/gmargari-24/airway_project/DGCI/logs/"
    "binary_128_2M_4k"
)

# Perplexity: 10, 20, 30, ..., 400
perplexity_values = range(5, 55, 5)

unique_labels = np.unique(labels)

# ------------------------------------------------------------
# Generate t-SNE projections
# ------------------------------------------------------------

for perplexity in perplexity_values:

    print(f"Running t-SNE with perplexity = {perplexity}...")

    # --------------------------------------------------------
    # t-SNE
    # --------------------------------------------------------
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        random_state=42,
        max_iter=1000
    )

    tsne_embedding = tsne.fit_transform(enc_output)

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------
    plt.figure(figsize=(10, 8))

    scatter = plt.scatter(
        tsne_embedding[:, 0],
        tsne_embedding[:, 1],
        c=labels,
        cmap="tab10",
        s=35,
        alpha=0.85
    )

    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.title(
        f"t-SNE projection of airway latent representations\n"
        f"perplexity = {perplexity}"
    )

    # --------------------------------------------------------
    # Legend
    # --------------------------------------------------------
    handles, _ = scatter.legend_elements()

    plt.legend(
        handles,
        [f"Cluster {c}" for c in unique_labels],
        title="Phenotype"
    )

    plt.tight_layout()

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------
    save_path = (
        f"{output_dir}/tSNE_clusters_perplexity_{perplexity}.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Saved: {save_path}")

print("All t-SNE projections generated.")

# UMAP and t-SNE of the clustered subjects good visualisation
Projects the same `enc_output` matrix that ward clustered into 2D, then colours each subject by its cluster in `labels`. Every cluster keeps the same colour and marker in both panels.

In [ ]:
# # --- UMAP and t-SNE of the clustered embeddings, coloured by cluster ----------
# import numpy as np
# import matplotlib.pyplot as plt
# import umap                                                     # umap-learn package
# from sklearn.manifold import TSNE

# save_dir = "/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k"

# X      = np.asarray(enc_output, dtype=np.float64)              # the exact matrix ward clustered
# labels = np.asarray(labels).astype(int)                        # from the clustering cell / labels.txt
# assert len(labels) == X.shape[0], "labels and embeddings have different lengths"

# # --- 1. project to 2D (same settings as Registration/umap.py and visualize_embeddings.py)
# umap_2d = umap.UMAP(n_neighbors=20, min_dist=0.0125, n_components=2,metric="euclidean", random_state=42).fit_transform(X)

# # perp    = min(20.0, (X.shape[0] - 1) / 3.0)                    # t-SNE needs perplexity < N/3
# tsne_2d = TSNE(n_components=2, perplexity=50, init="pca", random_state=0, max_iter=1000).fit_transform(X)

# # --- 2. one fixed colour + marker per cluster, identical in both panels ---------
# palette  = ["#2684f6", "#ab380b", "#11db94", "#f2b227", "#fc0c68", "#008300", "#4a3aa7", "#1c0404"]
# markers  = ["o"]             # shape backs up colour (colour-blind safe)
# clusters = np.unique(labels)
# assert len(clusters) <= len(palette), "more clusters than colours: extend palette and markers"
# style = {c: (palette[i], markers[i]) for i, c in enumerate(clusters)}
# order = sorted(clusters, key=lambda c: -(labels == c).sum())   # draw big clusters first, small ones on top

# fig, axes = plt.subplots(1, 2, figsize=(18, 8))
# for ax, coords, name in [(axes[0], umap_2d, "UMAP"), (axes[1], tsne_2d, "t-SNE")]:
#     for c in order:
#         m = labels == c
#         colour, marker = style[c]
#         ax.scatter(coords[m, 0], coords[m, 1], c=colour, marker=marker,
#                    s=40 if m.sum() > 20 else 90,               # enlarge small clusters so they stay visible
#                    alpha=0.85, edgecolors="white", linewidths=0.6,
#                    label=f"cluster {c} (n={m.sum()})")
#     ax.set_title(f"{name} of latent codes: {len(clusters)} ward clusters", fontsize=14)
#     ax.set_xlabel(f"{name} 1", fontsize=12)
#     ax.set_ylabel(f"{name} 2", fontsize=12)
#     ax.grid(color="#e5e5e5", linewidth=0.6)
#     ax.set_axisbelow(True)
#     for side in ["top", "right"]:
#         ax.spines[side].set_visible(False)

# handles, texts = axes[0].get_legend_handles_labels()
# pairs = sorted(zip(texts, handles), key=lambda t: int(t[0].split()[1]))   # legend in cluster-id order
# fig.legend([h for _, h in pairs], [t for t, _ in pairs], loc="center right", frameon=False, fontsize=11, markerscale=1.3)
# fig.tight_layout(rect=[0, 0, 0.88, 1])
# fig.savefig(f"{save_dir}/umap_tsne_clusters.png", dpi=300, bbox_inches="tight")
# plt.show()


In [ ]:
# # --- interactive version: hover a point to see which subject it is -------------
# import pandas as pd
# import plotly.express as px

# have_names = "case_names" in globals() and len(case_names) == len(labels)   # built in the tree-parse cell
# names = list(case_names) if have_names else [f"row_{i}" for i in range(len(labels))]

# for coords, name in [(umap_2d, "UMAP"), (tsne_2d, "t-SNE")]:
#     df = pd.DataFrame({"x": coords[:, 0], "y": coords[:, 1],
#                        "cluster": [f"cluster {c}" for c in labels], "case": names})
#     fig = px.scatter(df, x="x", y="y", color="cluster", hover_name="case",
#                      color_discrete_map={f"cluster {c}": style[c][0] for c in clusters},
#                      category_orders={"cluster": [f"cluster {c}" for c in clusters]},
#                      labels={"x": f"{name} 1", "y": f"{name} 2"},
#                      title=f"{name} of latent codes (hover for subject id)")
#     fig.update_traces(marker=dict(size=8, opacity=0.85, line=dict(width=0.5, color="white")))
#     fig.update_layout(width=850, height=650, plot_bgcolor="white")
#     fig.show()


# Morphological metrics projected onto the latents

Averages the 6 morphological metrics over the branches of each tree, aligns them to the
latent-code rows via `latent_codes.csv`, and colours the UMAP / t-SNE projections by each
metric in turn. Needs `morphology_projection.py` next to this notebook.


In [ ]:
# --- subjects in latent-code row order, then the mean metric per tree ---------
import numpy as np
import pandas as pd
import morphology_projection as mp                              # the helper module, same folder

latent_csv = "/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/latent_codes.csv"
cases_root = "/home/ids/gmargari-24/airway_project/airmorph_cases"
save_dir   = "/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/morphology"

mp.METRIC_NAMES = ["divergence", "length", "complexity", "tortuosity", "stenosis", "ectasia",]

latent_df = pd.read_csv(latent_csv)                             # same row order as latent_codes.npy
subjects  = latent_df["subject"].tolist()                       # 'AIIB23_30_R_mesh.mat', ...
assert len(subjects) == enc_output.shape[0], "csv rows != embedding rows — wrong file?"

metrics = mp.build_metric_table(subjects, cases_root, mp.METRIC_NAMES)   # (N, 6) means, row-aligned
metrics.to_csv(f"{save_dir}/mean_morphological_features.csv", index=False)   # keep for later stats
metrics.head()


In [ ]:
# --- interactive 3D counterparts of the UMAP and t-SNE loop plots -------------
import numpy as np                                                # arrays
import pandas as pd                                               # table for plotly
import plotly.express as px                                       # interactive 3D, same as the Kernel PCA figure
import umap                                                       # umap-learn package
from sklearn.manifold import TSNE                                 # t-SNE

X = np.asarray(enc_output, dtype=np.float64)                      # the exact matrix ward clustered
output_dir = "/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k"   # same folder as the 2D PNGs

neighbor_values   = range(5, 55, 5)                               # same as the 2D UMAP loop: 5, 10, ..., 50
perplexity_values = range(5, 55, 5)                               # same as the 2D t-SNE loop: 5, 10, ..., 50
assert len(neighbor_values) == len(perplexity_values), "ranges must pair up one-to-one"
show_inline = [(20, 20)]                                          # also display these settings here; all get saved

# --- same cluster colours and hover text as the Kernel PCA figure
labels   = np.asarray(labels).astype(int)                         # cluster id per subject, as integers
palette  = ["#2684f6", "#ab380b", "#11db94", "#f2b227", "#fc0c68", "#008300", "#4a3aa7", "#1c0404"]
clusters = np.unique(labels)                                      # sorted cluster ids
assert len(clusters) <= len(palette), "more clusters than colours: extend palette"
colour_map   = {f"cluster {c}": palette[i] for i, c in enumerate(clusters)}   # cluster name -> colour
cluster_text = [f"cluster {c}" for c in labels]                   # text so plotly uses discrete colours
hover_names  = list(subjects)                                     # subject ids on hover, from the morphology cell

projections_3d = {}                                               # (n_neighbors, perplexity) -> (umap, tsne), reused later


def cluster_figure_3d(coords, axis_name, title):
    """Interactive 3D scatter coloured by cluster, styled like the Kernel PCA figure."""
    df = pd.DataFrame({f"{axis_name} 1": coords[:, 0], f"{axis_name} 2": coords[:, 1],
                       f"{axis_name} 3": coords[:, 2], "cluster": cluster_text, "case": hover_names})
    fig = px.scatter_3d(df, x=f"{axis_name} 1", y=f"{axis_name} 2", z=f"{axis_name} 3",
                        color="cluster", hover_name="case", color_discrete_map=colour_map,
                        category_orders={"cluster": list(colour_map)}, title=title)
    fig.update_traces(marker=dict(size=4, opacity=0.85))          # same dots as the Kernel PCA figure
    fig.update_layout(width=850, height=700)                      # same size as the Kernel PCA figure
    return fig                                                    # caller saves and/or shows it


# pair the i-th UMAP setting with the i-th t-SNE setting (zip = walk both ranges together)
for n_neighbors, perplexity in zip(neighbor_values, perplexity_values):
    key = (n_neighbors, perplexity)                               # lookup key for this setting
    print(f"Running 3D n_neighbors = {n_neighbors}, perplexity = {perplexity}...")   # progress

    # n_components=3 is the only change from the 2D loops
    umap_3d_run = umap.UMAP(n_neighbors=n_neighbors, min_dist=0.0125, n_components=3,
                            metric="euclidean", random_state=42).fit_transform(X)
    tsne_3d_run = TSNE(n_components=3, perplexity=perplexity, init="pca",
                       random_state=42, max_iter=1000).fit_transform(X)
    projections_3d[key] = (umap_3d_run, tsne_3d_run)              # keep for the metric cell

    umap_fig = cluster_figure_3d(umap_3d_run, "UMAP", f"UMAP 3D, n_neighbors = {n_neighbors} (hover for subject id)")
    tsne_fig = cluster_figure_3d(tsne_3d_run, "t-SNE", f"t-SNE 3D, perplexity = {perplexity} (hover for subject id)")

    # cdn = load plotly.js from the web, so each file is KBs instead of ~4 MB
    umap_fig.write_html(f"{output_dir}/UMAP3D_clusters_neighbors_{n_neighbors}.html", include_plotlyjs="cdn")
    tsne_fig.write_html(f"{output_dir}/tSNE3D_clusters_perplexity_{perplexity}.html", include_plotlyjs="cdn")

    if key in show_inline:                                        # only a few inline, or the notebook gets huge
        umap_fig.show()                                           # render
        tsne_fig.show()                                           # render

print("All interactive 3D UMAP and t-SNE projections saved as HTML.")   # done

In [ ]:
# --- 6 UMAP + 6 t-SNE panels per metric, over the same ranges as the earlier loops ---
import os                                                         # one output folder per setting
import matplotlib.pyplot as plt                                   # close figures between runs
import umap                                                       # umap-learn package
from sklearn.manifold import TSNE

X = np.asarray(enc_output, dtype=np.float64)                      # the exact matrix ward clustered

neighbor_values   = range(5, 55, 5)                               # same as the UMAP loop: 5, 10, ..., 50
perplexity_values = range(5, 55, 5)                               # same as the t-SNE loop: 5, 10, ..., 50
assert len(neighbor_values) == len(perplexity_values), "ranges must pair up one-to-one"

# pair the i-th UMAP setting with the i-th t-SNE setting (zip = walk both ranges together)
for n_neighbors, perplexity in zip(neighbor_values, perplexity_values):
    print(f"Running n_neighbors = {n_neighbors}, perplexity = {perplexity}...")  # progress

    umap_run = umap.UMAP(n_neighbors=n_neighbors, min_dist=0.0125, n_components=2, metric="euclidean", random_state=42).fit_transform(X)  # same settings as the UMAP loop
    tsne_run = TSNE(n_components=2, perplexity=perplexity, init="pca",random_state=42, max_iter=1000).fit_transform(X)            # same settings as the t-SNE loop

    # folder name records both settings (:02d = zero-pad so folders sort 05, 10, 15...)
    run_dir = os.path.join(save_dir, f"umap_nn{n_neighbors:02d}_tsne_perp{perplexity:02d}")
    os.makedirs(run_dir, exist_ok=True)                           # create it (exist_ok = no error on re-run)

    table_run, figures = mp.project_metrics(umap_run, tsne_run, subjects, run_dir,root=cases_root, metric_names=mp.METRIC_NAMES)
    print(f"  saved {len(figures)} figures to {run_dir}")         # 12 panels + 2 grids per run
    plt.close("all")                                              # free memory; 10 runs = 140 figures

# --- default projections, still needed by the tree-count, kernel PCA and explorer cells ---
if "umap_2d" not in globals():                                    # reuse if already computed
    umap_2d = umap.UMAP(n_neighbors=20, min_dist=0.0125, n_components=2, metric="euclidean", random_state=42).fit_transform(X)   # original default view
if "tsne_2d" not in globals():                                    # reuse if already computed
    tsne_2d = TSNE(n_components=2, perplexity=50, init="pca", random_state=0, max_iter=1000).fit_transform(X)              # original default view

In [ ]:
# --- optional: per-cluster metric means, to name the phenotypes ----------------
summary = metrics[mp.METRIC_NAMES].copy()                       # drop the subject id column
summary["cluster"] = np.asarray(labels).astype(int)             # from the ward clustering cell

means = summary.groupby("cluster").mean()                       # average metric per phenotype
cluster_sizes = summary.groupby("cluster").size().rename("n")   # NOT `counts`: that name holds the tree-count table
print(pd.concat([cluster_sizes, means], axis=1).round(3))


## Whole-tree counts on the same projections

Two extra panels per projection — branch count and deepest generation — read from
`tree_counts.csv` (written by `count_branches_generations.py`). Run this after the
cell that builds `umap_2d` / `tsne_2d`.


In [ ]:
# --- 2 UMAP + 2 t-SNE panels per setting: branches and deepest generation ------
import os                                                         # one output folder per setting
import matplotlib.pyplot as plt                                   # close figures between runs
import umap                                                       # umap-learn package
from sklearn.manifold import TSNE
import morphology_projection as mp                                # restart the kernel to pick up edits

tree_counts_csv = "/home/ids/gmargari-24/airway_project/airmorph_cases/tree_counts.csv"

X = np.asarray(enc_output, dtype=np.float64)                      # the exact matrix ward clustered

neighbor_values   = range(5, 55, 5)                               # same as the UMAP loop: 5, 10, ..., 50
perplexity_values = range(5, 55, 5)                               # same as the t-SNE loop: 5, 10, ..., 50
assert len(neighbor_values) == len(perplexity_values), "ranges must pair up one-to-one"

if "projections" not in globals():                                # metrics cell wasn't run this session
    projections = {}                                              # start empty; the loop fills it

# pair the i-th UMAP setting with the i-th t-SNE setting (zip = walk both ranges together)
for n_neighbors, perplexity in zip(neighbor_values, perplexity_values):
    key = (n_neighbors, perplexity)                               # lookup key for this setting
    print(f"Running n_neighbors = {n_neighbors}, perplexity = {perplexity}...")  # progress

    if key not in projections:                                    # compute only if the metrics cell didn't keep it
        umap_new = umap.UMAP(n_neighbors=n_neighbors, min_dist=0.0125, n_components=2,
                             metric="euclidean", random_state=42).fit_transform(X)  # same settings as metrics loop
        tsne_new = TSNE(n_components=2, perplexity=perplexity, init="pca",
                        random_state=42, max_iter=1000).fit_transform(X)            # same settings as metrics loop
        projections[key] = (umap_new, tsne_new)                   # store so a re-run is instant
    umap_run, tsne_run = projections[key]                         # take the stored pair (tuple unpacking)

    # same folder as this setting's metric panels (:02d = zero-pad, 05 not 5)
    run_dir = os.path.join(save_dir, f"umap_nn{n_neighbors:02d}_tsne_perp{perplexity:02d}")
    os.makedirs(run_dir, exist_ok=True)                           # create it (exist_ok = no error if present)

    counts, count_figures = mp.project_tree_counts(umap_run, tsne_run, subjects,
                                                   tree_counts_csv, run_dir)
    print(f"  saved {len(count_figures)} figures to {run_dir}")   # 4 panels per run
    plt.close("all")                                              # free memory between runs

# counts.describe()                                               # sanity-check the spread

In [ ]:
# --- interactive 3D counterparts of the metric and tree-count panels ----------
import os                                                         # one output folder per setting
import numpy as np                                                # arrays
import pandas as pd                                               # align the colour values
import plotly.express as px                                       # interactive 3D, same as the Kernel PCA figure
import umap                                                       # umap-learn package
from sklearn.manifold import TSNE                                 # t-SNE

X = np.asarray(enc_output, dtype=np.float64)                      # the exact matrix ward clustered

neighbor_values   = range(5, 55, 5)                               # same as the 2D metric loop: 5, 10, ..., 50
perplexity_values = range(5, 55, 5)                               # same as the 2D metric loop: 5, 10, ..., 50
assert len(neighbor_values) == len(perplexity_values), "ranges must pair up one-to-one"
show_inline = [(20, 20)]                                          # also display these settings here; all get saved

labels       = np.asarray(labels).astype(int)                     # cluster id per subject, as integers
cluster_text = [f"cluster {c}" for c in labels]                   # shown on hover
hover_names  = list(subjects)                                     # subject ids on hover, from the morphology cell

# --- all 8 colour values in latent-row order: 6 morphology means + 2 tree counts
colour_table = pd.DataFrame({"subject": subjects})               # latent row order, from the morphology cell
colour_table = colour_table.merge(metrics, on="subject", how="left")   # 6 means (how="left" keeps row order)
colour_table = colour_table.merge(counts, on="subject", how="left")    # n_branches, max_generation
assert len(colour_table) == len(subjects), "merge changed the row count — duplicate subject ids?"
value_columns = [c for c in colour_table.columns if c != "subject"]    # the 8 things to colour by

if "projections_3d" not in globals():                             # 3D cluster cell wasn't run this session
    projections_3d = {}                                           # start empty; the loop fills it


def metric_figure_3d(coords, values, column, axis_name, title):
    """Interactive 3D scatter coloured by one metric; subjects without a value are grey."""
    x, y, z = f"{axis_name} 1", f"{axis_name} 2", f"{axis_name} 3"   # axis names (tuple unpacking)
    df = pd.DataFrame({x: coords[:, 0], y: coords[:, 1], z: coords[:, 2], column: values,
                       "cluster": cluster_text, "case": hover_names})
    has_value = df[column].notna()                                # rows with a measured value
    fig = px.scatter_3d(df[has_value], x=x, y=y, z=z, color=column, hover_name="case",
                        hover_data=["cluster"], color_continuous_scale="Viridis", title=title)
    missing = df[~has_value]                                      # rows with no value (~ = NOT)
    if len(missing) > 0:                                          # draw them grey so no subject disappears
        fig.add_scatter3d(x=missing[x], y=missing[y], z=missing[z], mode="markers",
                          marker=dict(color="lightgrey"), name="no value",
                          text=missing["case"], hoverinfo="text")
    fig.update_traces(marker=dict(size=4, opacity=0.85))          # same dots as the Kernel PCA figure
    fig.update_layout(width=850, height=700)                      # same size as the Kernel PCA figure
    return fig                                                    # caller saves and/or shows it


# pair the i-th UMAP setting with the i-th t-SNE setting, same folders as the 2D panels
for n_neighbors, perplexity in zip(neighbor_values, perplexity_values):
    key = (n_neighbors, perplexity)                               # lookup key for this setting
    print(f"Running 3D n_neighbors = {n_neighbors}, perplexity = {perplexity}...")   # progress

    if key not in projections_3d:                                 # compute only if the cluster cell didn't keep it
        umap_new = umap.UMAP(n_neighbors=n_neighbors, min_dist=0.0125, n_components=3,
                             metric="euclidean", random_state=42).fit_transform(X)
        tsne_new = TSNE(n_components=3, perplexity=perplexity, init="pca",
                        random_state=42, max_iter=1000).fit_transform(X)
        projections_3d[key] = (umap_new, tsne_new)                # store so a re-run is instant
    umap_3d_run, tsne_3d_run = projections_3d[key]                # take the stored pair (tuple unpacking)

    run_dir = os.path.join(save_dir, f"umap_nn{n_neighbors:02d}_tsne_perp{perplexity:02d}")  # :02d = zero-pad
    os.makedirs(run_dir, exist_ok=True)                           # create it (exist_ok = no error if present)

    for column in value_columns:                                  # 8 metrics × 2 methods = 16 files per setting
        values = colour_table[column].to_numpy(dtype=float)       # latent row order, NaN where missing
        if np.isnan(values).all():                                # nothing to colour by
            print(f"  skipped {column}: no values")               # say so instead of crashing
            continue

        umap_fig = metric_figure_3d(umap_3d_run, values, column, "UMAP",
                                    f"UMAP 3D, n_neighbors = {n_neighbors}: {column}")
        tsne_fig = metric_figure_3d(tsne_3d_run, values, column, "t-SNE",
                                    f"t-SNE 3D, perplexity = {perplexity}: {column}")

        # cdn = load plotly.js from the web, so each file is KBs instead of ~4 MB
        umap_fig.write_html(f"{run_dir}/UMAP3D_{column}.html", include_plotlyjs="cdn")
        tsne_fig.write_html(f"{run_dir}/tSNE3D_{column}.html", include_plotlyjs="cdn")

        if key in show_inline:                                    # only a few inline, or the notebook gets huge
            umap_fig.show()                                       # render
            tsne_fig.show()                                       # render

    print(f"  saved interactive 3D panels to {run_dir}")          # progress

print("All interactive 3D metric and tree-count panels saved as HTML.")   # done

## Per-cluster means of all 8 metrics

The 6 morphology means and the 2 whole-tree counts, summarised per ward cluster.
`cluster_summary` merges the tables on `subject`, so it can't silently misalign, and
`plot_cluster_heatmap` puts every metric on a common SD scale so they're comparable.


In [ ]:
# --- all 8 metrics, averaged within each cluster ------------------------------
import morphology_projection as mp

wide, detailed, merged = mp.cluster_summary(labels, metrics, counts, save_dir=save_dir)

print("\nmean of every metric per cluster:")
print(wide.to_string())                                         # n + the 8 means, one row per cluster

print("\nwith spread and valid counts:")
print(detailed.to_string())                                     # mean / sd / n per metric per cluster


In [ ]:
# --- the same table as a heatmap, every metric on a common SD scale ------------
heatmap_path = mp.plot_cluster_heatmap(merged, save_dir=save_dir)
print(heatmap_path)

# colour = how far that cluster sits from the cohort mean, in SD, so the 0-1 ratios
# and the 200-plus branch counts are comparable; the printed number is the real mean.

In [ ]:
# --- kernel PCA (RBF) of the ward-clustered codes: 2D and 3D views ---------------
from sklearn.decomposition import KernelPCA                     # PCA on RBF similarities
from sklearn.metrics import pairwise_distances                  # pairwise distances for the kernel width
import plotly.express as px                                     # interactive 3D that rotates without ipympl

import numpy as np                                              # arrays
import matplotlib.pyplot as plt                                 # static 2D/3D figure
import pandas as pd                                             # table for the plotly figure

# --- cluster colours and draw order, defined here (the cell that made them is commented out)
labels   = np.asarray(labels).astype(int)                       # cluster id per subject, as integers
palette  = ["#2684f6", "#ab380b", "#11db94", "#f2b227", "#fc0c68", "#008300", "#4a3aa7", "#1c0404"]
clusters = np.unique(labels)                                    # sorted cluster ids, e.g. 0..5
assert len(clusters) <= len(palette), "more clusters than colours: extend palette"
style    = {c: (palette[i], "o") for i, c in enumerate(clusters)}   # colour + plain dot per cluster
order    = sorted(clusters, key=lambda c: -(labels == c).sum()) # biggest first (minus sign = descending)

X_kpca = np.asarray(enc_output, dtype=np.float64)               # the exact matrix ward clustered (419, 128)

sq_dists   = pairwise_distances(X_kpca) ** 2                    # (419, 419) squared distances
kpca_gamma = 1.0 / np.median(sq_dists[sq_dists > 0])            # median rule: typical pair gets similarity ~0.37
print(f"median-rule gamma: {kpca_gamma:.3g}  (sklearn default 1/128 would be far too small here)")

kpca     = KernelPCA(kernel="rbf", gamma=kpca_gamma)            # keep all components to measure what the views show
kpca_all = kpca.fit_transform(X_kpca)                           # (419, n_components), largest spread first
kpca_3d  = kpca_all[:, :3]                                      # 3D view; its first 2 columns are the 2D view
eig      = kpca.eigenvalues_                                    # kernel variance per component
print(f"kernel variance shown: 2D {eig[:2].sum() / eig.sum():.1%}, 3D {eig[:3].sum() / eig.sum():.1%}")

# axis limits: 1st-99th percentile box, widened so every point of a small cluster stays in view
low  = np.percentile(kpca_3d, 1, axis=0)                        # lower crop per axis
high = np.percentile(kpca_3d, 99, axis=0)                       # upper crop per axis
for c in clusters:                                              # `clusters` from the UMAP/t-SNE cell
    m = labels == c                                             # members of cluster c
    if m.sum() < 10:                                            # small clusters must never be cropped away
        low  = np.minimum(low, kpca_3d[m].min(axis=0))          # extend the box down to include them
        high = np.maximum(high, kpca_3d[m].max(axis=0))         # and up
pad  = 0.05 * (high - low)                                      # small margin around the box
low  = low - pad                                                # padded lower limits
high = high + pad                                               # padded upper limits

fig = plt.figure(figsize=(10, 8))                               # single 2D panel, so half the old width
ax2  = fig.add_subplot(1, 1, 1)                                  # one panel (1 row, 1 column, slot 1)                                 # left: 2D view
for c in order:                                                 # `order` from the UMAP/t-SNE cell: big clusters first
    m = labels == c                                             # members of cluster c
    colour, marker = style[c]                                   # same colour + marker as the UMAP/t-SNE figure
    size = 40 if m.sum() > 20 else 90                           # enlarge small clusters so they stay visible
    ax2.scatter(kpca_3d[m, 0], kpca_3d[m, 1], c=colour, marker=marker, alpha=0.85,
                edgecolors="white", linewidths=0.6, label=f"cluster {c} (n={m.sum()})")

ax2.set_xlim(low[0], high[0])                                   # shared crop, x
ax2.set_ylim(low[1], high[1])                                   # shared crop, y
ax2.set_xlabel("Kernel PC1", fontsize=12)                       # largest-spread direction in kernel space
ax2.set_ylabel("Kernel PC2", fontsize=12)                       # second largest
ax2.set_title(f"Kernel PCA 2D: {len(clusters)} ward clusters", fontsize=14)
ax2.grid(color="#e5e5e5", linewidth=0.6)                        # same light grid as the UMAP/t-SNE figure
ax2.set_axisbelow(True)                                         # grid behind the points

handles, texts = ax2.get_legend_handles_labels()                # legend entries from the 2D panel
pairs = sorted(zip(texts, handles), key=lambda t: int(t[0].split()[1]))  # legend in cluster-id order
fig.legend([h for _, h in pairs], [t for t, _ in pairs], loc="center right", frameon=False, fontsize=11)
fig.tight_layout(rect=[0, 0, 0.8, 1])                           # leave room for the legend on the right
fig.savefig(f"{save_dir}/kernel_pca_2d_clusters.png", dpi=300, bbox_inches="tight")  # new name: no 3D inside
plt.show()                                                      # render

# --- interactive 3D: rotate with the mouse, hover a point to see the subject ------
kpca_df = pd.DataFrame({"KPC1": kpca_3d[:, 0], "KPC2": kpca_3d[:, 1], "KPC3": kpca_3d[:, 2],
                        "cluster": [f"cluster {c}" for c in labels],   # text so plotly uses discrete colours
                        "case": subjects})                              # `subjects` from the morphology cell
fig3d = px.scatter_3d(kpca_df, x="KPC1", y="KPC2", z="KPC3", color="cluster", hover_name="case",
                      color_discrete_map={f"cluster {c}": style[c][0] for c in clusters},
                      category_orders={"cluster": [f"cluster {c}" for c in clusters]},
                      title="Kernel PCA 3D of latent codes (hover for subject id)")
fig3d.update_traces(marker=dict(size=4, opacity=0.85))          # smaller markers suit 3D
fig3d.update_layout(width=850, height=700)                      # similar size to the 2D plotly figures
fig3d.show()                                                    # render

In [ ]:
# --- Kernel PCA coloured by each of the 8 metrics: 2D (static) + 3D (interactive) ---
import numpy as np                                              # arrays
import pandas as pd                                             # align the colour values
import matplotlib.pyplot as plt                                 # static 2D figures
import plotly.express as px                                     # interactive 3D figures

show_3d_inline = True                                           # False = only save the 3D HTML files

# --- all 8 colour values in latent-row order: 6 morphology means + 2 tree counts
colour_table = pd.DataFrame({"subject": subjects})              # latent row order, from the morphology cell
colour_table = colour_table.merge(metrics, on="subject", how="left")   # 6 means (how="left" keeps row order)
colour_table = colour_table.merge(counts, on="subject", how="left")    # n_branches, max_generation
assert len(colour_table) == len(subjects), "merge changed the row count — duplicate subject ids?"
value_columns = [c for c in colour_table.columns if c != "subject"]    # the 8 things to colour by
cluster_text  = [f"cluster {c}" for c in labels]                # shown on hover in 3D

for column in value_columns:                                    # one 2D + one 3D figure per metric
    values    = colour_table[column].to_numpy(dtype=float)      # latent row order, NaN where missing
    has_value = ~np.isnan(values)                               # subjects with a measured value (~ = NOT)
    missing   = ~has_value                                      # subjects without one
    if not has_value.any():                                     # nothing to colour by
        print(f"skipped {column}: no values")                   # say so instead of crashing
        continue

    # colour scale spans the 1st-99th percentile, so one extreme tree can't wash out the rest
    vmin = np.percentile(values[has_value], 1)                  # bottom of the colour scale
    vmax = np.percentile(values[has_value], 99)                 # top of the colour scale

    # --- 2D: same crop, grid, dots and outline as the cluster plot above
    fig = plt.figure(figsize=(10, 8))                           # same size as the cluster plot
    ax2 = fig.add_subplot(1, 1, 1)                              # one panel
    ax2.scatter(kpca_3d[missing, 0], kpca_3d[missing, 1], c="lightgrey", alpha=0.85,
                edgecolors="white", linewidths=0.6, label="no value")   # grey, so no subject disappears
    present = np.flatnonzero(has_value)                         # row numbers that have a value
    present = present[np.argsort(values[present])]              # low values first, high values drawn on top
    points = ax2.scatter(kpca_3d[present, 0], kpca_3d[present, 1], c=values[present], cmap="viridis",
                         vmin=vmin, vmax=vmax, alpha=0.85, edgecolors="white", linewidths=0.6)
    fig.colorbar(points, ax=ax2, extend="both", label=column)   # value scale (extend = arrows mark clipped values)

    ax2.set_xlim(low[0], high[0])                               # same crop as the cluster plot, x
    ax2.set_ylim(low[1], high[1])                               # same crop, y
    ax2.set_xlabel("Kernel PC1", fontsize=12)                   # axis names match the cluster plot
    ax2.set_ylabel("Kernel PC2", fontsize=12)
    ax2.set_title(f"Kernel PCA 2D: {column}", fontsize=14)      # metric name
    ax2.grid(color="#e5e5e5", linewidth=0.6)                    # same light grid
    ax2.set_axisbelow(True)                                     # grid behind the points
    if missing.any():                                           # only explain grey when grey dots exist
        ax2.legend(loc="upper right", frameon=False)            # "no value" entry

    fig.tight_layout()                                          # trim margins
    fig.savefig(f"{save_dir}/kernel_pca_2d_{column}.png", dpi=300, bbox_inches="tight")
    plt.show()                                                  # render

    # --- 3D: same look as the interactive cluster figure above
    kpca_df = pd.DataFrame({"KPC1": kpca_3d[:, 0], "KPC2": kpca_3d[:, 1], "KPC3": kpca_3d[:, 2],
                            column: values, "cluster": cluster_text, "case": subjects})
    fig3d = px.scatter_3d(kpca_df.loc[has_value], x="KPC1", y="KPC2", z="KPC3", color=column,
                          hover_name="case", hover_data=["cluster"],       # hover: subject, cluster, value
                          color_continuous_scale="Viridis", range_color=[vmin, vmax],
                          title=f"Kernel PCA 3D: {column} (hover for subject id)")
    if missing.any():                                           # grey dots for subjects without a value
        grey = kpca_df.loc[missing]                             # just those rows
        fig3d.add_scatter3d(x=grey["KPC1"], y=grey["KPC2"], z=grey["KPC3"], mode="markers",
                            marker=dict(color="lightgrey"), name="no value",
                            text=grey["case"], hoverinfo="text")
        fig3d.update_layout(legend=dict(x=0, y=1))              # legend top-left, clear of the colour bar
    fig3d.update_traces(marker=dict(size=4, opacity=0.85))      # same dots as the cluster figure
    fig3d.update_layout(width=850, height=700)                  # same size as the cluster figure
    fig3d.write_html(f"{save_dir}/kernel_pca_3d_{column}.html", include_plotlyjs="cdn")   # keep an interactive copy
    if show_3d_inline:                                          # 8 interactive figures below the cell
        fig3d.show()                                            # render

print(f"Saved 2D PNGs and 3D HTML files for {len(value_columns)} metrics to {save_dir}")   # done

In [ ]:
# --- box plots per cluster, all 8 metrics: agglomerative (ward) ---------------------
import numpy as np                                                    # arrays
import pandas as pd                                                   # tables keyed by subject
import matplotlib.pyplot as plt                                       # plotting
import morphology_projection as mp                                    # only for the tree-count fallback


def plot_cluster_boxplots(cluster_labels, metrics, counts, method_name, out_dir):
    """One box per cluster for each of the 8 metrics; tick labels show n, mean and variance."""
    table = metrics.merge(counts, on="subject", how="left")          # 6 means + 2 counts (left = keep row order)
    table["cluster"] = np.asarray(cluster_labels).astype(int)        # cluster id per subject, same row order
    metric_columns = [c for c in table.columns if c not in ("subject", "cluster")]  # the 8 metrics
    clusters = np.unique(table["cluster"])                           # sorted cluster ids

    n_cols = 4                                                       # 4 panels per row
    n_rows = int(np.ceil(len(metric_columns) / n_cols))              # 8 metrics -> 2 rows
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 6 * n_rows))
    axes = np.atleast_1d(axes).ravel()                               # flat list of panels (ravel = 2D -> 1D)

    stat_rows = []                                                   # mean / variance per cluster, for printing
    for ax, metric in zip(axes, metric_columns):                     # one panel per metric
        groups = []                                                  # values of each cluster
        tick_text = []                                               # label under each box
        for c in clusters:                                           # one box per cluster
            values = table.loc[table["cluster"] == c, metric].dropna().to_numpy(dtype=float)
            groups.append(values)                                    # data for this box
            mean = values.mean() if len(values) else np.nan          # cluster mean
            var = values.var(ddof=1) if len(values) > 1 else np.nan  # sample variance (ddof=1 = divide by n-1)
            tick_text.append(f"cluster {c}\nn={len(values)}\nμ={mean:.3g}\nσ²={var:.3g}")
            stat_rows.append({"metric": metric, "cluster": c, "n": len(values),
                              "mean": mean, "variance": var, "sd": np.sqrt(var)})

        positions = np.arange(1, len(clusters) + 1)                  # x position of each box
        ax.boxplot(groups, positions=positions, widths=0.6,
                   showmeans=True, meanline=False,                   # draw the mean as a marker
                   meanprops=dict(marker="D", markerfacecolor="red", markeredgecolor="black", markersize=7),
                   medianprops=dict(color="black", linewidth=1.5),   # median = black line
                   flierprops=dict(marker="o", markersize=3, alpha=0.4))  # outliers, faint

        for pos, values in zip(positions, groups):                   # overlay every subject as a dot
            jitter = np.random.default_rng(0).uniform(-0.15, 0.15, len(values))  # spread dots sideways
            ax.scatter(pos + jitter, values, s=6, alpha=0.3, color="grey", zorder=0)  # zorder=0 = behind boxes

        ax.set_xticks(positions)                                     # one tick per box
        ax.set_xticklabels(tick_text, fontsize=9)                    # n, mean and variance under each box
        ax.set_title(metric, fontsize=13)                            # which metric this panel shows
        ax.grid(axis="y", color="#e5e5e5", linewidth=0.6)            # light horizontal grid
        ax.set_axisbelow(True)                                       # grid behind the boxes

    for ax in axes[len(metric_columns):]:                            # hide unused panels, if any
        ax.set_visible(False)

    fig.suptitle(f"{method_name}: metrics per cluster (red diamond = mean, line = median)", fontsize=16)
    fig.tight_layout(rect=[0, 0, 1, 0.96])                           # leave room for the title
    path = f"{out_dir}/boxplots_{method_name.replace(' ', '_').lower()}.png"  # e.g. boxplots_k-means.png
    fig.savefig(path, dpi=300, bbox_inches="tight")                  # write the PNG
    plt.show()                                                       # render

    stats = pd.DataFrame(stat_rows)                                  # long table: metric x cluster
    stats_wide = stats.pivot(index="cluster", columns="metric", values=["mean", "variance"])  # one row per cluster
    print(path)                                                      # where the figure went
    return stats_wide                                                # so you can compare with your table


# --- run it for ward -------------------------------------------------------------------
if "counts" not in globals():                                        # tree-count cell not run this session
    tree_counts_csv = "/home/ids/gmargari-24/airway_project/airmorph_cases/tree_counts.csv"
    counts = mp.build_tree_count_table(subjects, tree_counts_csv)    # subject + n_branches + max_generation

ward_box_stats = plot_cluster_boxplots(labels, metrics, counts, "Agglomerative", save_dir)
ward_box_stats.round(3)                                              # mean + variance per cluster

In [ ]:
# ----- CELL A: 1_clustering.ipynb (agglomerative / ward, k = 6) ---------------------
import build_explorer as bx                                          # the explorer builder
 
ward = bx.Explorer("Airway latent space explorer", subjects)         # title and latent-row order
ward.set_metrics(colour_table)                                       # 'subject' + the 8 metric columns
 
m = "Agglomerative clustering"                                       # method tab name
ward.add(m, "Kernel PCA", labels, coords_2d=kpca_3d[:, :2], coords_3d=kpca_3d, axes="Kernel PC")
 
two = globals().get("projections", {})                               # (nn, perp) -> 2D (umap, tsne)
three = globals().get("projections_3d", {})                          # (nn, perp) -> 3D (umap, tsne)
keys = sorted(set(two) | set(three))                                 # every setting either loop computed
assert keys, "run the 2D and 3D UMAP / t-SNE loops first"
print(f"exporting {len(keys)} settings: {[k[0] for k in keys]}")     # 5, 10, ..., 50
 
for n_neighbors, perplexity in keys:                                 # one entry per setting
    umap_2d_run, tsne_2d_run = two.get((n_neighbors, perplexity), (None, None))
    umap_3d_run, tsne_3d_run = three.get((n_neighbors, perplexity), (None, None))
    if umap_2d_run is not None or umap_3d_run is not None:           # UMAP menu entry
        ward.add(m, "UMAP", labels, coords_2d=umap_2d_run, coords_3d=umap_3d_run,
                 axes="UMAP ", parameter="n_neighbors", setting=n_neighbors)
    if tsne_2d_run is not None or tsne_3d_run is not None:           # t-SNE menu entry
        ward.add(m, "t-SNE", labels, coords_2d=tsne_2d_run, coords_3d=tsne_3d_run,
                 axes="t-SNE ", parameter="perplexity", setting=perplexity)
 
ward.dump(f"{save_dir}/explorer_ward.json")                          # picked up by CELL B
